<b><font size="6" color="#E8800A">Week 7a · Probability and Naive Bayes</font></b><br>
<b><font size="4">Bayes' theorem, three assumptions, and what they cost</font></b><br>

The classification baseline you already have learned a boundary.
This week takes a different route to the same problem: model how the data was
*generated*, then invert that model with **Bayes' theorem** to get a probability
for each class.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Bayes' theorem</font>](#bayes)
* [<font color='#E8800A'>A Bayes classifier without assumptions</font>](#full)
* [<font color='#E8800A'>The three naive assumptions</font>](#naive)
* [<font color='#E8800A'>Implementations of Naive Bayes</font>](#implementations)
* [<font color='#E8800A'>Fitting Gaussian Naive Bayes</font>](#fit)
* [<font color='#E8800A'>How Gaussian Naive Bayes predicts</font>](#predict)
* [<font color='#E8800A'>Bayes without the independence assumption</font>](#non-naive)
* [<font color='#E8800A'>More than two classes</font>](#multiclass)
* [<font color='#E8800A'>The settings of Naive Bayes</font>](#nb-params)
* [<font color='#E8800A'>Is the grid worth running?</font>](#nested)
* [<font color='#E8800A'>The grid search</font>](#search)
* [<font color='#E8800A'>The number we stand behind</font>](#closing)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>Bayes' theorem</font> <a class="anchor" id="bayes"></a>
[Back to TOC](#toc)

The probability that two events $A$ and $B$ both happen, $P(A \cap B)$, can be built in either order. It is the probability of $B$ times the probability of $A$ once $B$ is known, written $P(A \mid B)$, or the same with the two events swapped:

$$P(A \cap B) = P(A \mid B)\,P(B) = P(B \mid A)\,P(A).$$

Dividing the second equality by $P(B)$, which must be above zero, gives **Bayes' theorem**, $P(A \mid B) = P(B \mid A)\,P(A)/P(B)$.

For an athlete, $A$ is the event that her outcome $y$ is class $c$, with $y = 1$ if she won and $y = 0$ if not. $B$ is the event that her features take the values $x = (x_1, \dots, x_p)$, one value for each of the $p$ columns:

$$\underbrace{P(y = c \mid x)}_{\text{posterior}} = \frac{\overbrace{P(x \mid y = c)}^{\text{likelihood}}\;\overbrace{P(y = c)}^{\text{prior}}}{\underbrace{P(x)}_{\text{evidence}}}.$$

The **prior** is the probability of class $c$ before any feature is read. The **likelihood** is the probability of these feature values among the athletes of class $c$. Their product is the **joint** $P(x, y = c)$, the probability of these values and class $c$ together.

The **evidence** $P(x) = \sum_k P(x \mid y = k)\,P(y = k)$ is the probability of these values over all athletes: the sum of the joints over the classes $k$. The **posterior** is the probability of class $c$ once the features are read.

The evidence is the same for every class, so dividing by it makes the joints sum to one without changing which is largest. The classifier predicts the class with the largest posterior.

For a numeric feature $P(x \mid y = c)$ is a density rather than a probability, and the theorem holds with densities in its place.

In [ ]:
import sys
from pathlib import Path

# course_helpers.py and preprocessing.py sit in this folder, next to the notebook.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.datasets import load_digits
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler

from course_helpers import (
    PLOT_BLUE,
    PLOT_ORANGE,
    Scoreboard,
    record,
    result_context,
    results_before,
)

random_state = 33
np.random.seed(random_state)  # For reproducibility
figsize = (7, 5)

# Fold-safe encoding means a rare level can occur only in validation, so this
# message would repeat once per scored fold.
warnings.filterwarnings(
    "ignore",
    message="Found unknown categories.*during transform",
    category=UserWarning,
)

from preprocessing import (
    CachedSearchCV,
    FoldPreprocessor,
    PreparedEstimator,
    load_recipe,
    outer_folds,
    outer_splitter,
    prepared_folds,
)
classification_recipe_log = "../../logs/week_04_feature_work_classification_log.json"


# <font color='#E8800A'>A Bayes classifier without assumptions</font> <a class="anchor" id="full"></a>
[Back to TOC](#toc)

To apply the theorem to one athlete, each class needs two numbers. The **prior** is the share of the class among the training athletes. The **likelihood** is the probability of his whole profile, every column at once, among the athletes of that class. The steps below compute both for one athlete by counting, with no assumption.

The **chain rule** splits a whole profile into one column at a time, by applying $P(A \cap B) = P(A \mid B)\,P(B)$ again and again. For a winner and three of the columns:

$$P(\text{Competition}, \text{Sex}, \text{Region} \mid \text{won}) = P(\text{Competition} \mid \text{won}) \times P(\text{Sex} \mid \text{Competition}, \text{won}) \times P(\text{Region} \mid \text{Competition}, \text{Sex}, \text{won}).$$

Each factor is a share inside a smaller group. The first is the share of the winners who entered his competition. The second is the share of those winners who have his sex. The third is the share of the winners who match him on both who come from his region.

With all $p$ columns, and $x_j$ his value in column $j$, the rule gives $p$ factors:

$$P(x_1, \dots, x_p \mid y = c) = P(x_1 \mid y = c)\,P(x_2 \mid x_1, y = c) \cdots P(x_p \mid x_1, \dots, x_{p-1}, y = c).$$

Factor $j$ is the share of the athletes of class $c$ who match him on the first $j - 1$ columns that also match him on column $j$. The rule is exact: it rewrites the likelihood without simplifying it.

__Step 1:__ Load the classification spine and split it.

In [ ]:
champions, classification_recipe = load_recipe(
    "../../data/interim/champions.csv",
    classification_recipe_log,
)
# RecordID names each row, so it becomes the index.
champions = champions.set_index("RecordID")
X_churn = champions.loc[:, classification_recipe["categorical"] + classification_recipe["numeric"]].copy()
y_churn = champions["Outcome"]

X_churn_train, X_churn_test, y_churn_train, y_churn_test = train_test_split(
    X_churn, y_churn, test_size=0.2, stratify=y_churn, random_state=random_state,
)
print(f"{len(X_churn_train):,} training athletes, {len(X_churn_test):,} test athletes,"
      f" positive rate {y_churn_train.mean():.4f}")

__Step 2:__ Prepare the athletes, and fix the folds every score is read on.

In [ ]:
churn_prep = FoldPreprocessor(classification_recipe).fit(X_churn_train, y_churn_train)
X_churn_train_ready = churn_prep.transform(X_churn_train)
X_churn_test_ready = churn_prep.transform(X_churn_test)

# The ten folds every validation score is read on.
churn_folds = outer_splitter("classification", random_state=random_state)
churn_prepared = prepared_folds(
    FoldPreprocessor(classification_recipe), churn_folds,
    X_churn_train, y_churn_train, n_jobs=4)

__Step 3:__ Pick a training winner with a value in every categorical column and a profile no other training athlete shares, and show his profile.

In [ ]:
categorical = classification_recipe["categorical"]
complete = X_churn_train[categorical].dropna()
# A winner whose whole profile no other training athlete has.
alone = ~complete.duplicated(keep=False)
athlete = complete[alone & (y_churn_train.loc[complete.index] == 1)].iloc[0]
athlete

__Step 4:__ The prior: the share of each class among the training athletes.

In [ ]:
y_churn_train.value_counts(normalize=True).sort_index().round(4)

**0.3959 of the training athletes did not win and 0.6041 won**: before his profile is read, these are his probabilities of each class.

__Step 5:__ The likelihood, by the chain rule: each row adds one of his columns, and each cell counts the training athletes of that class who match him on every column so far.

In [ ]:
rows = ["the class alone"] + [f"and {column} = {athlete[column]}" for column in categorical]
chain = pd.DataFrame(index=rows)
for label, name in [(1, "winners"), (0, "other athletes")]:
    matching = X_churn_train.loc[y_churn_train == label, categorical]
    counts = [len(matching)]
    for column in categorical:
        matching = matching[matching[column] == athlete[column]]
        counts.append(len(matching))
    chain[name] = counts
chain

**The last row counts 1 winner and 0 other athletes.** Each factor of the chain rule is a count divided by the count above it. Among the winners the first factor is $200/1933$, the share who entered his competition, and the second $171/200$, the share of those who have his sex.

Multiplied together, every count between the first and the last cancels. Without assumptions, then, the likelihood is the last count over the first: $1/1933$ among the winners and $0/1267$ among the other athletes, the share of each class with exactly his profile. The priors are the first counts over the 3,200 training athletes, so the winners' joint is $(1/1933)(1933/3200) = 1/3200$ and the other athletes' joint is 0. His posterior of winning is $(1/3200)/(1/3200 + 0) = 1$, and any prior above zero gives the same 1. The one training athlete with his profile is he himself: the counts have memorised him.

__Step 6:__ A test athlete: count the training athletes of each class who share her whole profile.

In [ ]:
# The first test athlete with a value in every categorical column.
newcomer = X_churn_test[categorical].dropna().iloc[0]
for label, name in [(1, "winners"), (0, "other athletes")]:
    matching = X_churn_train.loc[y_churn_train == label, categorical]
    for column in categorical:
        matching = matching[matching[column] == newcomer[column]]
    print(f"training {name} with this profile: {len(matching)}")

In [ ]:
# Every test athlete with no gap: does any training athlete share this profile?
seen = set(X_churn_train[categorical].dropna().itertuples(index=False))
test_profiles = X_churn_test[categorical].dropna()
unseen = sum(profile not in seen for profile in test_profiles.itertuples(index=False))
print(f"test athletes whose profile no training athlete shares: {unseen} of {len(test_profiles)}")
print(f"possible combinations of the {len(categorical)} columns' levels:"
      f" {X_churn_train[categorical].nunique().prod():,}, for {len(X_churn_train):,} training athletes")

**1 training winner and 2 other training athletes share this profile**, so by the cancellation of Step 5 this athlete's posterior of winning would be one third, decided by 3 athletes. For 431 of the 759 test athletes with no gap, no training athlete shares the profile at all. Both counts are then 0, the likelihood is 0 in each class, and the posterior is 0 divided by 0. The 12 columns allow 873,600 combinations of levels against 3,200 training athletes, so most profiles were never seen.

__Step 7:__ The full likelihood keeps every relation between the columns. Measure how strongly the numeric columns move together inside each class.

In [ ]:
# Within each class, the three pairs of numeric columns with the largest correlation.
numeric = classification_recipe["numeric"]
for label in [0, 1]:
    within = X_churn_train.loc[y_churn_train == label, numeric].corr()
    pairs = within.where(np.triu(np.ones(within.shape, dtype=bool), k=1)).stack().dropna()
    print(f"class {label}")
    print(pairs[pairs.abs().nlargest(3).index].round(3).to_string(), "\n")

**0.783 among the athletes who did not win and 0.726 among the winners: the strongest pair in each class**, both with `Train bf competition`, and all six pairs printed are positive. Athletes who train a lot train more of *everything*, so the training-volume columns rise and fall together, and knowing one tells a good deal about the next.

__Step 8:__ Count the cells a full table would need, and the memory it would take.

In [ ]:
# One cell per combination of values: each level of a categorical column, and
# ten intervals for each numeric column, at 8 bytes a cell.
sizes = [X_churn_train[column].nunique() for column in categorical] + [10] * len(numeric)
used = [3, 6, len(categorical), 16, 20, len(sizes)]
table_size = pd.DataFrame({"columns used": used,
                           "cells per class": [np.prod(sizes[:count], dtype=float) for count in used]})
table_size["GB, both classes"] = 2 * 8 * table_size["cells per class"] / 1e9
print(table_size.to_string(index=False, float_format="{:.3g}".format))
print(f"\ntraining athletes to fill them: {len(X_churn_train):,}")

**The 12 categorical columns need 8.74e+05 cells per class. All 26 columns, with the numeric ones cut into ten intervals, need 8.74e+19 cells per class and 1.4e+12 GB for both classes.** Each added column multiplies the table by its number of values, so the size grows as a product and not as a sum. It reaches 140 GB at 16 columns and 1.4e+06 GB at 20. The 3,200 training athletes can fill at most 3,200 of those cells.

# <font color='#E8800A'>The three naive assumptions</font> <a class="anchor" id="naive"></a>
[Back to TOC](#toc)

Naive Bayes replaces the full likelihood with pieces that a few thousand athletes can estimate, through three assumptions about the athletes of each class.

**1. Independence given the class.** Once the class is known, the other columns tell nothing more about column $j$:

$$P(x_j \mid x_1, \dots, x_{j-1}, y = c) = P(x_j \mid y = c), \qquad j = 2, \dots, p.$$

Each factor of the chain rule then asks the whole class instead of a shrinking group. For the athlete of Steps 3 to 5, the first factor among the winners stays the share of all winners in his competition. The second becomes the share of all winners of his sex, the third the share of all winners from his region, and so on. The likelihood becomes a product of $p$ factors, one per column:

$$P(x \mid y = c) = \prod_{j=1}^{p} P(x_j \mid y = c).$$

**2. Equal weight.** Taking logarithms turns the product into a sum, and the log posterior is

$$\log P(y = c \mid x) = \log P(y = c) + \sum_{j=1}^{p} \log P(x_j \mid y = c) - \log P(x).$$

Every column adds one term with weight 1. Weighting column $j$ by $w_j$ would mean the term $w_j \log P(x_j \mid y = c)$, its factor raised to the power $w_j$.

**3. A known distribution.** Within each class, each column follows a distribution of a known form with few parameters, so each factor $P(x_j \mid y = c)$ is a formula rather than a table of counts. Which distribution depends on the implementation.

With the three in place, the model needs only the class shares and a few numbers per column per class. Each is an average over the training athletes, computed in one pass over them.

# <font color='#E8800A'>Implementations of Naive Bayes</font> <a class="anchor" id="implementations"></a>
[Back to TOC](#toc)

The implementations in `sklearn.naive_bayes` differ in one thing: the distribution of assumption 3, the form of $P(x_j \mid y = c)$. Bayes' theorem, the product over the columns and the prior are the same in all of them.

| implementation | $P(x_j \mid y = c)$ | what it estimates | data it suits |
|---|---|---|---|
| `GaussianNB` | a normal density with mean $\mu_{c,j}$ and variance $\sigma^2_{c,j}$ | $\mu_{c,j}$ and $\sigma^2_{c,j}$, the mean and the variance of feature $j$ among the training rows of class $c$ | numeric features, close to normal within each class |
| `BernoulliNB` | $\theta_{c,j}^{\,x_j}(1 - \theta_{c,j})^{1 - x_j}$, for $x_j$ equal to 0 or 1 | $\theta_{c,j} = (N_{c,j} + \alpha)/(N_c + 2\alpha)$, the probability that $x_j = 1$ in class $c$: $N_c$ rows of class $c$, $N_{c,j}$ of them with $x_j = 1$, and $\alpha$ (default 1) the smoothing parameter, added to every count so that no value gets probability zero | binary features, such as one-hot levels (values above `binarize`, default 0, count as 1) |
| `CategoricalNB` | $\theta_{c,j,t}$, when $x_j$ is level $t$ | $\theta_{c,j,t} = (N_{c,j,t} + \alpha)/(N_c + \alpha n_j)$, with $N_{c,j,t}$ rows of class $c$ at level $t$ and $n_j$ levels of feature $j$ | categorical features coded as integers $0, \dots, n_j - 1$ |
| `MultinomialNB` | $\theta_{c,j}^{\,x_j}$, for a count $x_j$; the multinomial coefficient is equal in every class and is dropped | $\theta_{c,j} = (N_{c,j} + \alpha)/(N_c + \alpha p)$, feature $j$'s share of the counts in class $c$: $N_{c,j}$ is the total of feature $j$ over the rows of class $c$, and $N_c$ the total over all $p$ features | non-negative counts, such as word counts |

This notebook uses `GaussianNB`, which gives every column a normal density within each class, and the next step checks that assumption on the prepared columns.

__Step 9:__ Check GaussianNB's assumption on the prepared columns.

In [ ]:
# A column that takes only 0 and 1 cannot follow a normal distribution; for the
# others, the skew within each class says how far from symmetric they are.
prepared_names = list(churn_prep.get_feature_names_out())
winners = y_churn_train.to_numpy() == 1
shape = pd.DataFrame({
    "distinct_values": [np.unique(column).size for column in X_churn_train_ready.T],
    "skew_class_0": pd.DataFrame(X_churn_train_ready[~winners]).skew().to_numpy(),
    "skew_class_1": pd.DataFrame(X_churn_train_ready[winners]).skew().to_numpy(),
}, index=prepared_names)
zero_one = [set(np.unique(column)) == {0.0, 1.0} for column in X_churn_train_ready.T]
print(f"{sum(zero_one)} of the {len(prepared_names)} prepared columns take only the values 0 and 1\n")
print(shape[[not flag for flag in zero_one]].round(2).to_string())

**18 of the 28 prepared columns take only the values 0 and 1.** Within a class each of them is a Bernoulli variable, the case `BernoulliNB` models, and `GaussianNB` applies a normal density to it anyway. A normal distribution also has skew 0. Among the ten numeric columns, `Previous attempts` reaches a skew of 3.21 and 4.57 in the two classes, and `Sand training` 3.33 and 2.30. `Train bf competition` comes closest to symmetric, at -0.40 and -0.10.

# <font color='#E8800A'>Fitting Gaussian Naive Bayes</font> <a class="anchor" id="fit"></a>
[Back to TOC](#toc)

`GaussianNB` gives column $j$ in class $c$ the normal density

$$P(x_j \mid y = c) = \frac{1}{\sqrt{2\pi\sigma^2_{c,j}}}\exp\!\left(-\frac{(x_j - \mu_{c,j})^2}{2\sigma^2_{c,j}}\right),$$

a bell curve centred on the class mean $\mu_{c,j}$, wider as the class variance $\sigma^2_{c,j}$ grows. The factor in front makes the area under the curve one. Fitting estimates the prior and these two numbers by counting and averaging over the training athletes of each class:

$$P(y=c) = \frac{n_c}{n}, \qquad \mu_{c,j} = \frac{1}{n_c}\sum_{i:\,y_i=c} x_{ij}, \qquad \sigma^2_{c,j} = \frac{1}{n_c}\sum_{i:\,y_i=c} \left(x_{ij}-\mu_{c,j}\right)^2 .$$

Here $n$ is the number of training athletes and $n_c$ the number in class $c$. The value of column $j$ for athlete $i$ is $x_{ij}$, and each sum runs over the athletes $i$ whose class $y_i$ is $c$. The prior is the share of class $c$, $\mu_{c,j}$ the mean of column $j$ in that class, and $\sigma^2_{c,j}$ the mean squared distance from that mean.

`var_smoothing` adds a small constant to every variance so that none is zero. Nothing is fitted iteratively: no loss is minimised and no estimate is revised step by step.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html'>sklearn.naive_bayes.GaussianNB(*, priors=None, var_smoothing=1e-09)</a>

**Definition:**
Gaussian Naive Bayes. Fits one mean and one variance per feature per class, then applies Bayes' theorem assuming the features are conditionally independent given the class.

**Main Parameters:**
- `priors` (default `None`): $P(y)$, the belief about each class before seeing any features. `None` estimates it from the training frequencies.
- `var_smoothing` (default `1e-9`): a fraction of the largest feature variance added to every variance, so a class with a near-constant feature cannot produce a division by zero.

**Main Attributes (after fitting):**
- `class_prior_`: the priors actually used.
- `theta_`: one mean per feature per class.
- `var_`: one variance per feature per class, smoothing included.

**Main Methods:**
- `fit(X, y)`
- `predict(X)`
- `predict_proba(X)`: the posteriors themselves.
</div>

__Step 10:__ Score Naive Bayes on the folds, and open the log.

In [ ]:
nb_model = PreparedEstimator(FoldPreprocessor(classification_recipe), GaussianNB())
nb_cv = cross_validate(
    nb_model, X_churn_train, y_churn_train, cv=churn_folds,
    scoring="f1", return_train_score=True)
nb_fitted = clone(nb_model).fit(X_churn_train, y_churn_train)

In [ ]:
churn_log = []
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": "GaussianNB()",
                   "train": nb_cv["train_score"],
                   "validation": nb_cv["test_score"], "model": nb_fitted},
       X_churn_test, y_churn_test, metric=f1_score, label="F1")

**F1 0.7765 on the validation folds and 0.8177 on the test rows.** The train F1, 0.7840, sits just above the validation F1: the model fits the athletes it was fitted on hardly better than those it never saw.

# <font color='#E8800A'>How Gaussian Naive Bayes predicts</font> <a class="anchor" id="predict"></a>
[Back to TOC](#toc)

A prediction is Bayes' theorem with the fitted numbers in place of the counts. The steps below read each fitted attribute in turn and follow the athlete of Step 3 to his posterior.

__Step 11:__ `class_prior_`: the prior of each class the model uses.

In [ ]:
gaussian = nb_fitted.estimator_
gaussian.class_prior_.round(4)

**0.3959 and 0.6041**: the class shares Step 4 printed.

__Step 12:__ `theta_` and `var_`: one mean and one variance for each class and prepared column.

In [ ]:
fitted_numbers = pd.DataFrame({
    "mean, class 0": gaussian.theta_[0], "mean, class 1": gaussian.theta_[1],
    "variance, class 0": gaussian.var_[0], "variance, class 1": gaussian.var_[1],
}, index=nb_fitted.preprocessor_.get_feature_names_out())
fitted_numbers.round(3)

Each row holds $\mu_{c,j}$ and $\sigma^2_{c,j}$ for one prepared column, computed over the training athletes of each class; the variance includes the constant `var_smoothing` adds. For a column of 0 and 1 the mean is the share of the class at that level: 0.015 of the athletes who did not win and 0.110 of the winners have `Mental preparation_True`. For `Train bf competition` the means are -0.522 and 0.247.

__Step 13:__ One column for him: the normal density of `Train bf competition` at his value, in each class.

In [ ]:
# His row, prepared the way the fitted model prepares it.
athlete_ready = nb_fitted.preprocessor_.transform(X_churn_train.loc[[athlete.name]])[0]
column = prepared_names.index("Train bf competition")
value = athlete_ready[column]
print(f"his prepared value: {value:.3f}")
for label in [0, 1]:
    mean = gaussian.theta_[label, column]
    variance = gaussian.var_[label, column]
    height = np.exp(-(value - mean) ** 2 / (2 * variance)) / np.sqrt(2 * np.pi * variance)
    print(f"class {label}: mean {mean:.3f}, standard deviation {np.sqrt(variance):.3f},"
          f" density at his value {height:.4f}")

In [ ]:
grid = np.linspace(X_churn_train_ready[:, column].min(), X_churn_train_ready[:, column].max(), 300)
plt.figure(figsize=figsize)
for label, name, colour in [(0, "did not win (class 0)", PLOT_BLUE), (1, "won (class 1)", PLOT_ORANGE)]:
    mean, variance = gaussian.theta_[label, column], gaussian.var_[label, column]
    curve = np.exp(-(grid - mean) ** 2 / (2 * variance)) / np.sqrt(2 * np.pi * variance)
    plt.plot(grid, curve, color=colour, linewidth=2, label=name)
plt.axvline(value, color="grey", linestyle=":", linewidth=1.5, label="his value")
plt.xlabel("Train bf competition (prepared)")
plt.ylabel("normal density")
plt.title("The normal density of each class for Train bf competition")
plt.legend()
plt.show()

**At his value, 0.154, the winners' curve stands at 0.6493 and the other athletes' at 0.3547.** The density is the height of each class's curve at his value, so the taller curve, here the winners', is the class this column favours. A density is not a probability: it is a height per unit of the column, and a narrow curve can rise above 1.

__Step 14:__ All the columns: multiply his 28 densities in each class, then apply Bayes' theorem.

In [ ]:
# One normal density per class and column, from the fitted means and variances.
density = (np.exp(-(athlete_ready - gaussian.theta_) ** 2 / (2 * gaussian.var_))
           / np.sqrt(2 * np.pi * gaussian.var_))
likelihood = density.prod(axis=1)          # one factor per column, each counted once
joint = likelihood * gaussian.class_prior_
evidence = joint.sum()
posterior = joint / evidence

print(f"likelihood  P(x | y)      {likelihood}")
print(f"joint       P(x | y) P(y) {joint}")
print(f"evidence    P(x)          {evidence:.6f}")
print(f"posterior   P(y | x)      {posterior.round(5)}")
print(f"predict_proba             {gaussian.predict_proba(athlete_ready.reshape(1, -1))[0].round(5)}")
print(f"predict                   {gaussian.predict(athlete_ready.reshape(1, -1))[0]}")
print(f"his outcome               {y_churn_train[athlete.name]}")

**His posterior of winning is 0.99978, by hand and from `predict_proba` alike, and he won.** The likelihoods, products of 28 densities, are $3.80 \times 10^{-4}$ for not winning and 1.11 for winning: like a density, a product of densities can exceed 1. Times the priors they give the joints, $1.51 \times 10^{-4}$ and 0.67265, whose sum is the evidence, 0.672801. Dividing each joint by it gives the two posteriors, 0.00022 and 0.99978, and `predict` returns 1, the class with the larger posterior.

# <font color='#E8800A'>Bayes without the independence assumption</font> <a class="anchor" id="non-naive"></a>
[Back to TOC](#toc)

Can a Gaussian classifier improve by modelling how the columns vary together? Keep a normal density in each class and priors from the class shares, but allow covariance between columns. The covariance may belong to each class or be shared; that choice changes another assumption as well. The columns of a class are then modelled together by the **multivariate normal density**:

$$P(x \mid y = c) = (2\pi)^{-p/2}\,\lvert\Sigma_c\rvert^{-1/2}\exp\!\left(-\tfrac{1}{2}(x - \mu_c)^{\top}\Sigma_c^{-1}(x - \mu_c)\right).$$

Here $x$ is the column of an athlete's $p$ values and $\mu_c$ the column of their means among the training athletes of class $c$. The mark $^{\top}$ turns a column into a row. $\Sigma_c$ is the $p \times p$ covariance matrix of class $c$. Its diagonal holds the variance of each column, and entry $(j, k)$ the covariance of columns $j$ and $k$, which measures how the two move together. $\lvert\Sigma_c\rvert$ is its determinant, and with $(2\pi)^{-p/2}$ it makes the density integrate to one.

$\Sigma_c^{-1}$ is the inverse of $\Sigma_c$, so the density exists only when $\Sigma_c$ can be inverted. The quadratic form $(x - \mu_c)^{\top}\Sigma_c^{-1}(x - \mu_c)$ is a squared distance from the class mean that allows for how the columns vary and move together.

When every covariance off the diagonal is zero, the density is the product of the $p$ normal densities of `GaussianNB`. One full $\Sigma_c$ for each class is **quadratic discriminant analysis** (QDA). One $\Sigma$ shared by the two classes is **linear discriminant analysis** (LDA).

__Step 15:__ Check which covariance can be inverted: each class's own, or the one shared by both classes.

In [ ]:
# A covariance matrix can be inverted only if its rank equals p, the number of columns.
p = X_churn_train_ready.shape[1]
centred = []
for label in [0, 1]:
    rows = X_churn_train_ready[y_churn_train.to_numpy() == label]
    fixed = [name for name, column in zip(prepared_names, rows.T) if column.std() == 0]
    print(f"class {label}: rank {np.linalg.matrix_rank(np.cov(rows, rowvar=False))} of {p};"
          f" columns that never vary: {fixed}")
    centred.append(rows - rows.mean(axis=0))
pooled = np.cov(np.vstack(centred), rowvar=False)
print(f"shared by both classes: rank {np.linalg.matrix_rank(pooled)} of {p}")

**Rank 27 of 28 for the winners' covariance, against 28 of 28 for the other athletes' and for the covariance shared by both classes.** `No coach_True` never varies among the winners, so its row and its column of $\Sigma_1$ are zero and $\Sigma_1$ cannot be inverted. The winners' density then cannot be evaluated, and QDA cannot be fitted as it stands. The shared covariance can be inverted, which is LDA.

<div class="alert alert-block alert-info">

**With one shared covariance, the log posterior odds are linear.** Scikit-learn estimates the shared $\Sigma$ from all the training athletes, each measured from the mean of her own class. In the ratio of the two posteriors the evidence cancels, and with one $\Sigma$ so do the factor in front of the two densities and the term $x^{\top}\Sigma^{-1}x$ of their quadratic forms. What remains is linear in $x$:

$$\log\frac{P(y = 1 \mid x)}{P(y = 0 \mid x)} = x^{\top}\Sigma^{-1}(\mu_1 - \mu_0) - \tfrac{1}{2}(\mu_1 + \mu_0)^{\top}\Sigma^{-1}(\mu_1 - \mu_0) + \log\frac{P(y = 1)}{P(y = 0)}.$$

Here $\mu_1$ and $\mu_0$ are the mean vectors of the winners and of the other athletes, and the last term is the log of the prior odds. The first term is a weighted sum of the columns with weights $w = \Sigma^{-1}(\mu_1 - \mu_0)$, and the other two form a constant $b$: the form $w^{\top}x + b$ that logistic regression fits to the labels directly.

</div>

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.discriminant_analysis.LinearDiscriminantAnalysis.html'>sklearn.discriminant_analysis.LinearDiscriminantAnalysis(solver='svd', shrinkage=None, priors=None, n_components=None, store_covariance=False, tol=0.0001, covariance_estimator=None)</a>

**Definition:**
Linear discriminant analysis. Fits one mean vector per class and one covariance matrix shared by the classes, then applies Bayes' theorem, so the boundary between the classes is linear.

**Main Parameters:**
- `priors` (default `None`): $P(y)$. `None` uses the class shares of the training rows.

**Main Attributes (after fitting):**
- `means_`: one mean vector per class, the $\mu_c$.
- `coef_` and `intercept_`: with two classes, the weights $w$ and the constant $b$ of the log posterior odds.

**Main Methods:**
- `fit(X, y)`
- `predict(X)`
- `predict_proba(X)`: the posteriors.
</div>

__Step 16:__ Score LDA on the same folds, and log it. LDA allows the columns to vary together and shares one covariance across classes. Its score beside Naive Bayes measures those two changes together. The model is `LinearDiscriminantAnalysis()`.

In [ ]:
lda_model = PreparedEstimator(FoldPreprocessor(classification_recipe),
                              ...)
lda_cv = cross_validate(
    lda_model, X_churn_train, y_churn_train, cv=churn_folds,
    scoring="f1", return_train_score=True)
lda_fitted = clone(lda_model).fit(X_churn_train, y_churn_train)

In [ ]:
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": "LinearDiscriminantAnalysis()",
                   "train": lda_cv["train_score"],
                   "validation": lda_cv["test_score"], "model": lda_fitted},
       X_churn_test, y_churn_test, metric=f1_score, label="F1")

**LDA scores 0.8495 on the validation folds and 0.8485 on the test rows, against 0.7765 and 0.8177 for Naive Bayes.** Both give each class a normal density and take the priors from the class shares. LDA allows covariance between columns and gives both classes one shared covariance, where Naive Bayes gives each class its own variances. The score gap measures those changes together; it does not isolate the cost of independence.

# <font color='#E8800A'>More than two classes</font> <a class="anchor" id="multiclass"></a>
[Back to TOC](#toc)

Nothing in Bayes' theorem cared that there were two classes. Compute a
posterior for each class and pick the largest, and the method works unchanged for
ten. What does *not* survive unchanged is the scoring.

__Step 17:__ Load the handwritten-digits dataset and make it deliberately
imbalanced, so the averaging schemes have something to disagree about.

<div class="alert alert-block alert-info">

**A dataset built to order.** `Outcome` has two classes, and `load_digits`
has ten, the handwritten digits 0 to 9, which the next cell makes uneven on
purpose.

Nothing measured on digits joins the course scoreboard: its column is the
`champions` outcome, and an F1 on handwritten digits does not compare with it.

</div>

In [ ]:
digits_X, digits_y = load_digits(return_X_y=True, as_frame=True)

# Keep every row for digits 0-5 and only 15% for 6-9, so the classes are
# uneven: constructed on purpose, so the metrics below can differ.
rng = np.random.RandomState(random_state)
selected = []
for digit in range(10):
    positions = np.where(digits_y == digit)[0]
    share = 1.0 if digit < 6 else 0.15
    selected.extend(rng.choice(positions, int(len(positions) * share), replace=False))

selected = np.sort(selected)
X_digits, y_digits = digits_X.iloc[selected], digits_y.iloc[selected]
print(np.bincount(y_digits))

__Step 18:__ Split the digits and fold the training part, then predict each fold's validation part and the test rows with Gaussian Naive Bayes.

In [ ]:
d_train, d_test, dy_train, dy_test = train_test_split(
    X_digits, y_digits, test_size=0.2, stratify=y_digits, random_state=random_state,
)
# The scaler is fitted on each fold's training part only.
digit_folds = outer_splitter("classification", random_state=random_state)
digit_prepared = prepared_folds(StandardScaler(), digit_folds, d_train, dy_train)
digit_folds_predicted = [GaussianNB().fit(fold["X_fit"], fold["y_fit"]).predict(fold["X_score"])
                         for fold in digit_prepared]
digit_scaler = StandardScaler().fit(d_train)
digit_predictions = GaussianNB().fit(digit_scaler.transform(d_train), dy_train).predict(
    digit_scaler.transform(d_test))

__Step 19:__ Score the same predictions four different ways.

In [ ]:
scheme_rows = []
for name, average in [("accuracy", None), ("micro F1", "micro"),
                      ("macro F1", "macro"), ("weighted F1", "weighted")]:
    if average is None:
        validation = [accuracy_score(fold["y_score"], predicted)
                      for fold, predicted in zip(digit_prepared, digit_folds_predicted)]
        test = accuracy_score(dy_test, digit_predictions)
    else:
        validation = [f1_score(fold["y_score"], predicted, average=average)
                      for fold, predicted in zip(digit_prepared, digit_folds_predicted)]
        test = f1_score(dy_test, digit_predictions, average=average)
    scheme_rows.append({"score": name, "validation": np.mean(validation), "test": test})
pd.DataFrame(scheme_rows).set_index("score").round(3)

<div class="alert alert-block alert-info">

**Four numbers for one model, and they are not interchangeable.**

| what | how it is built |
|---|---|
| accuracy | share of rows classified correctly |
| **micro** F1 | pool every class's true positives, false positives and false negatives, then compute F1 once |
| **macro** F1 | compute F1 per class, then take a plain average |
| **weighted** F1 | per-class F1, averaged with weights equal to class size |

**Micro F1 equals accuracy exactly**, 0.596 on the folds and 0.513 on the test
rows, and that is not a coincidence on this data: when every observation gets
exactly one predicted label, each wrong label counts once as a
false positive, for the class predicted, and once as a false negative, for the
true class, so pooled precision, pooled recall and micro F1 all equal accuracy.

**Macro is the lowest**, 0.546 on the folds, and **weighted the highest**,
0.644. Macro gives every digit the same vote, and weighted gives each digit a
vote in proportion to its rows, so weighted above macro means the model scores
better, on balance, on the common digits than on the rare ones.

</div>

__Step 20:__ Score each digit on its own, on the validation folds and the test rows.

In [ ]:
per_class = pd.DataFrame({
    "cv_F1": np.mean([f1_score(fold["y_score"], predicted, average=None, labels=range(10))
                      for fold, predicted in zip(digit_prepared, digit_folds_predicted)], axis=0),
    "test_F1": f1_score(dy_test, digit_predictions, average=None),
    "test_rows": np.bincount(dy_test),
}, index=[f"digit {digit}" for digit in range(10)])
print(per_class.round(2).sort_values("cv_F1").to_string())

Per-class validation F1 runs from **0.18** for digit 8 to **0.99** for
digit 0. Every averaged number above hides that spread, and three of the four
rare digits, 8, 9 and 7, sit at the bottom. Each rare digit has five test rows, so
its test column rests on five predictions.

A confusion matrix shows *where* the mistakes go: how often each true digit is
predicted as each other digit.

In [ ]:
# The validation parts of the ten folds, summed: every training digit twice.
matrix = sum(confusion_matrix(fold["y_score"], predicted, labels=range(10))
             for fold, predicted in zip(digit_prepared, digit_folds_predicted))

plt.figure(figsize=figsize)
plt.imshow(matrix, cmap="Blues")
plt.colorbar(label="validation predictions")
plt.xlabel("predicted digit")
plt.ylabel("true digit")
plt.title("True digit against predicted digit, validation folds summed")
plt.xticks(range(10)); plt.yticks(range(10))
plt.show()

# <font color='#E8800A'>The settings of Naive Bayes</font> <a class="anchor" id="nb-params"></a>
[Back to TOC](#toc)

`GaussianNB` has two settings, and each changes a term of Bayes' theorem:
`var_smoothing` adds a constant to the variances inside the likelihood, and
`priors` sets the prior. Neither is a learning rate or a penalty. The third
setting acts on the model's output: `CalibratedClassifierCV`, through its
`method`, replaces the model's probabilities with ones learned on rows the model
was not fitted on.

Each step below changes **one** setting and holds the others at
their defaults, prints validation and test F1 side by side, and logs its best value. The
steps are here to show what each setting does; the choice is made by one grid
search over every combination of the settings tested, after this section.
In a project you would go straight to that grid.

**`var_smoothing`: a floor under the variances.** The normal density divides by $\sigma^2_{c,j}$. A feature that barely varies within a class therefore gets a narrow spike: very large at the class mean, almost zero a short distance away. That one factor can then outweigh all the others.

`GaussianNB` uses $\sigma^2_{c,j} + \epsilon$ in place of every $\sigma^2_{c,j}$, with

$$\epsilon = v \times \max_j \operatorname{Var}(x_j).$$

Here $v$ is the value of `var_smoothing`. $\operatorname{Var}(x_j)$ is the variance of column $j$ over all the rows the model is fitted on, both classes together, and the maximum runs over the columns. The default, $v = 10^{-9}$, adds a billionth of the largest column variance.

`No coach_True` never varies among the winners, as the covariance check printed, so its variance in that class is zero. Without $\epsilon$, its density could not be evaluated.

**Prediction:** nothing changes until $\epsilon$ is comparable to the variances themselves; beyond that, $\epsilon$ swamps the differences between the classes and the model collapses onto its prior.

__Step 21:__ Walk `var_smoothing` from a trillionth up to ten. The values to try are `[1e-12, 1e-9, 1e-6, 1e-3, 1e-1, 1.0, 10.0]`; at `1e-9` it is at its default.

In [ ]:
smoothing_values = ...

In the loop, the model is `GaussianNB(var_smoothing=value)`. Each value changes only the constant added to every variance, and the sweep shows how F1, accuracy and recall move as that constant grows from negligible to larger than the variances themselves.

In [ ]:
# F1 chooses; accuracy and recall show what the model is doing.
smoothing_rows = []
for value in smoothing_values:
    model = ...
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in churn_prepared]
    labels = clone(model).fit(X_churn_train_ready, y_churn_train).predict(X_churn_test_ready)
    row = {"configuration": f"GaussianNB(var_smoothing={value:g})", "var_smoothing": value,
           "train": [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"]))
                     for fold, fitted in zip(churn_prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], fitted.predict(fold["X_score"]))
                          for fold, fitted in zip(churn_prepared, fold_models)],
           "model": model}
    row["cv_F1"], row["test_F1"] = np.mean(row["validation"]), f1_score(y_churn_test, labels)
    for name, metric in [("accuracy", accuracy_score), ("recall", recall_score)]:
        row[f"cv_{name}"] = np.mean([metric(fold["y_score"], fitted.predict(fold["X_score"]))
                                     for fold, fitted in zip(churn_prepared, fold_models)])
        row[f"test_{name}"] = metric(y_churn_test, labels)
    smoothing_rows.append(row)
smoothing_sweep = pd.DataFrame(smoothing_rows)
smoothing_sweep.set_index("var_smoothing")[["cv_F1", "test_F1", "cv_accuracy",
                                             "test_accuracy", "cv_recall", "test_recall"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = smoothing_sweep.loc[smoothing_sweep["cv_F1"].idxmax()]
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": best["configuration"],
                   "train": best["train"], "validation": best["validation"],
                   "model": clone(best["model"]).fit(X_churn_train_ready, y_churn_train)},
       X_churn_test_ready, y_churn_test, "one setting changed", metric=f1_score, label="F1")

In [ ]:
figure, panels = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)
for panel, name in zip(panels, ["F1", "accuracy", "recall"]):
    panel.plot(smoothing_sweep["var_smoothing"], smoothing_sweep[f"cv_{name}"], marker="o",
               markersize=8, linewidth=2, color=PLOT_BLUE, label="validation (ten folds)")
    panel.plot(smoothing_sweep["var_smoothing"], smoothing_sweep[f"test_{name}"], marker="o",
               markersize=8, linewidth=2, linestyle="--", color=PLOT_ORANGE, label="test")
    panel.axvline(1e-9, color="grey", linestyle=":", linewidth=1, label="the default")
    panel.set_xscale("log")
    panel.set_title(name)
    panel.set_xlabel("var_smoothing (log scale)")
panels[0].set_ylabel("score")
panels[0].legend()
plt.show()

**Half of the prediction survives.** F1 does not sit still until the
smoothing is large. On the folds it runs **0.7585, 0.7765, 0.7822** from 1e-12
through the default to 1e-6, falls to **0.7258** at 1e-3 and recovers to
**0.7724** at 1.0. The best value on the folds is **1e-6**, not the default, and
it joins the log.

At **10.0** the model collapses as predicted. Recall reaches **1.0000** in both
columns, so the model calls almost every athlete a winner. Its accuracy on the
folds, **0.6095**, sits beside the positive rate Step 1 printed, 0.6041. Its F1,
0.7558, is close to the 0.7585 of the working model at 1e-12: F1 alone barely
separates a collapsed model from a working one.

**`priors`: the prior, set by hand.** The prior $P(y = c)$ is the first term of the log posterior. Left at `None`, it is $\hat\pi_c$, the share of class $c$ among the training athletes.

Other values $q_c$ multiply the numerator of class $c$ by $q_c/\hat\pi_c$, a factor that differs between the classes and not between athletes. The posterior odds of winning are then multiplied by one constant, the same for every athlete, which moves the boundary. `theta_` and `var_` stay as fitted, because each class estimates them from its own rows.

**Prediction:** a larger prior on winning makes the model call more athletes winners, so recall rises and precision falls, and a smaller one reverses the trade.

__Step 22:__ Set the prior five ways, and read five metrics for each setting. The values to try are `[None, [0.8, 0.2], [0.6, 0.4], [0.5, 0.5], [0.2, 0.8]]`; `None` estimates the priors from the training frequencies, the default.

In [ ]:
prior_values = ...

In the loop, the model is `GaussianNB(priors=prior)`. Each value changes only the prior, which moves the boundary between the classes, and the sweep shows how precision and recall trade as it moves.

In [ ]:
scores = {"precision": precision_score, "recall": recall_score,
          "F0.5": lambda truth, labels: fbeta_score(truth, labels, beta=0.5),
          "F1": f1_score,
          "F2": lambda truth, labels: fbeta_score(truth, labels, beta=2)}
prior_rows = []
for prior in prior_values:
    model = ...
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in churn_prepared]
    labels = clone(model).fit(X_churn_train_ready, y_churn_train).predict(X_churn_test_ready)
    row = {"configuration": f"GaussianNB(priors={prior})", "priors": str(prior),
           "train": [f1_score(fold["y_fit"], fitted.predict(fold["X_fit"]))
                     for fold, fitted in zip(churn_prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], fitted.predict(fold["X_score"]))
                          for fold, fitted in zip(churn_prepared, fold_models)],
           "model": model}
    for name, metric in scores.items():
        row[f"cv_{name}"] = np.mean([metric(fold["y_score"], fitted.predict(fold["X_score"]))
                                     for fold, fitted in zip(churn_prepared, fold_models)])
        row[f"test_{name}"] = metric(y_churn_test, labels)
    prior_rows.append(row)
prior_sweep = pd.DataFrame(prior_rows)
print(prior_sweep[["priors", *[f"{prefix}_{name}" for name in scores
                                for prefix in ("cv", "test")]]].round(4).to_string(index=False))
print()
for name in ["F0.5", "F1", "F2"]:
    best_at = prior_sweep[f"cv_{name}"].idxmax()
    print(f"best priors on validation {name:4s}: {prior_sweep.loc[best_at, 'priors']:10s}"
          f" {prior_sweep.loc[best_at, f'cv_{name}']:.4f}")

In [ ]:
# The sweep's winner joins the log.
best = prior_sweep.loc[prior_sweep["cv_F1"].idxmax()]
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": best["configuration"],
                   "train": best["train"], "validation": best["validation"],
                   "model": clone(best["model"]).fit(X_churn_train_ready, y_churn_train)},
       X_churn_test_ready, y_churn_test, "one setting changed", metric=f1_score, label="F1")

<div class="alert alert-block alert-warning">

**The prior moves precision and recall as the prediction said, and the three $F_\beta$ columns
disagree about where to leave it.**

On the folds `[0.8, 0.2]`, believing most athletes do *not* win, gives precision
**0.7364** and recall **0.8514**: the model has become more reluctant to call a
winner, and buys precision with the recall it gives up. `[0.2, 0.8]` does the
reverse: precision **0.6810**, recall **0.9188**. The last three lines name the
winner of each $F_\beta$ on the validation folds: `[0.8, 0.2]` for $F_{0.5}$,
`[0.6, 0.4]` for $F_1$ and `[0.2, 0.8]` for $F_2$. The two extreme priors win
the two extremes of $\beta$: $F_{0.5}$ weighs precision more than recall, and $F_2$
weighs recall more than precision.

On $F_1$ the five priors span only **0.7748 to 0.7771**, and `[0.6, 0.4]` joins
the log at 0.7771, against the default's 0.7765. Each $F_\beta$ chose a different
prior, so the prior the log keeps depends on the metric it is scored with.

</div>

**Calibration (`method`): a third setting, on the probabilities.** F1 scores the predicted labels, but it does not tell us whether the probabilities are reliable. A model is **calibrated** when, among the athletes it gives a probability near $p$ of winning, a share near $p$ win.

The sweep reads F1 beside two measures. **Accuracy** is the share of athletes whose predicted label is right. **Confidence** is the probability the model gives the label it predicts, averaged over the athletes. Confidence above accuracy signals that the model is too sure of its predictions on average. Agreement between these averages alone does not establish calibration across the probability range.

`CalibratedClassifierCV` wraps the whole preprocessing and Naive Bayes pair. It corrects the probabilities in four steps:

1. It splits the rows it is fitted on, in the sweep the training part of a fold, into 5 internal folds.
2. It fits preprocessing, including feature selection, and Naive Bayes on four of them.
3. It transforms the fifth with that fitted preprocessing, then learns how the model's probability of winning, $s$, maps to the share of athletes who actually win. These rows did not select features or fit the classifier.
4. It repeats this five times, once for each fold, and averages the five corrected probabilities.

The map has one of two shapes. `sigmoid` (Platt scaling) is an S-shaped curve with two fitted numbers, $p' = 1/(1 + \exp(a s + b))$, where $p'$ is the corrected probability and $a$ and $b$ are fitted on the fifth fold. The corrected probability rises with $s$ when $a$ is negative. `isotonic` is any curve that never goes down as $s$ grows. `None`, the first value of the sweep, keeps the plain model. The label is the class with the larger corrected probability, so F1 can move too.

**Prediction:** both methods bring confidence down toward accuracy, and F1 moves much less than confidence.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibratedClassifierCV.html'>sklearn.calibration.CalibratedClassifierCV(estimator=None, *, method='sigmoid', cv=None, n_jobs=None, ensemble='auto')</a>

**Definition:**
Probability calibration with cross-validation. Fits copies of a classifier on internal training parts and learns, on each held-out part, a map from its probability to the observed share of the class.

**Main Parameters:**
- `estimator` (default `None`, which means `LinearSVC(random_state=0)`): the classifier whose probabilities are calibrated, here the `PreparedEstimator` pair containing preprocessing and `GaussianNB()`.
- `method` (default `'sigmoid'`): `'sigmoid'` (Platt scaling), `'isotonic'` or `'temperature'`. The notebook compares the first two.
- `cv` (default `None`): the internal folds. `None` means 5 stratified folds.

**Main Attributes (after fitting):**
- `calibrated_classifiers_`: the fitted pairs of classifier and calibrator, five with the defaults.

**Main Methods:**
- `fit(X, y)`
- `predict(X)`: the class with the largest corrected probability.
- `predict_proba(X)`: the average of the five corrected probabilities.
</div>

__Step 23:__ Calibrate the probabilities, and read F1 beside accuracy and confidence. Each method corrects the probabilities of the same Naive Bayes, and the sweep shows how far confidence falls toward accuracy and how little F1 moves. The methods to try are `[None, "sigmoid", "isotonic"]`; `None` leaves Naive Bayes uncalibrated, the default.

In [ ]:
calibration_values = ...

In [ ]:
# Calibration must split raw rows before fitting any learned preprocessing.
calibration_data = [
    {"X_fit": X_churn_train.iloc[fit_at], "y_fit": y_churn_train.iloc[fit_at],
     "X_score": X_churn_train.iloc[score_at], "y_score": y_churn_train.iloc[score_at]}
    for fit_at, score_at in churn_folds.split(X_churn_train, y_churn_train)
]
base_nb = PreparedEstimator(FoldPreprocessor(classification_recipe), GaussianNB())
calibration_rows = []
for method in calibration_values:
    model = clone(base_nb).set_params(calibration=method)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in calibration_data]
    fitted = clone(model).fit(X_churn_train, y_churn_train)
    row = {"configuration": "GaussianNB()" if method is None
           else f"CalibratedClassifierCV(GaussianNB(), method='{method}')",
           "method": str(method),
           "train": [f1_score(fold["y_fit"], trained.predict(fold["X_fit"]))
                     for fold, trained in zip(calibration_data, fold_models)],
           "validation": [f1_score(fold["y_score"], trained.predict(fold["X_score"]))
                          for fold, trained in zip(calibration_data, fold_models)],
           "model": model}
    row["cv_F1"] = np.mean(row["validation"])
    row["test_F1"] = f1_score(y_churn_test, fitted.predict(X_churn_test))
    row["cv_accuracy"] = np.mean([accuracy_score(fold["y_score"], trained.predict(fold["X_score"]))
                                  for fold, trained in zip(calibration_data, fold_models)])
    row["test_accuracy"] = accuracy_score(y_churn_test, fitted.predict(X_churn_test))
    # Confidence: the probability the model gives the label it predicts.
    row["cv_confidence"] = np.mean([trained.predict_proba(fold["X_score"]).max(axis=1).mean()
                                    for fold, trained in zip(calibration_data, fold_models)])
    row["test_confidence"] = fitted.predict_proba(X_churn_test).max(axis=1).mean()
    calibration_rows.append(row)
calibration_sweep = pd.DataFrame(calibration_rows)
calibration_sweep.set_index("method")[["cv_F1", "test_F1", "cv_accuracy", "test_accuracy",
                                       "cv_confidence", "test_confidence"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = calibration_sweep.loc[calibration_sweep["cv_F1"].idxmax()]
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": best["configuration"],
                   "train": best["train"], "validation": best["validation"],
                   "model": clone(best["model"]).fit(X_churn_train, y_churn_train)},
       X_churn_test, y_churn_test, "one setting changed", metric=f1_score, label="F1")

**Uncalibrated, Naive Bayes is 0.9455 confident on the validation folds and right 0.6944 of the time.** Its confidence exceeds its accuracy, so it is too sure of its predictions on average. Calibration brings confidence to 0.6942 with `sigmoid` and 0.7409 with `isotonic`, beside accuracies of 0.7198 and 0.7373. F1 moves far less, from 0.7765 to 0.7920 with `sigmoid` and 0.7977 with `isotonic`, which joins the log. On the test rows `sigmoid` scores higher, 0.8196 against 0.8155.

# <font color='#E8800A'>Is the grid worth running?</font> <a class="anchor" id="nested"></a>
[Back to TOC](#toc)

The three sweep winners each changed one setting. A grid fits every combination instead: 105 here, the plain model's 35 pairs of smoothing and prior, and the calibrated model's 70, the same pairs under each of the two methods. `PreparedEstimator` applies each calibration method to the complete preprocessing/model pair. `CachedSearchCV` shares fitted preprocessing only when the training rows and labels are identical. Each calibration training split therefore fits its own recipe.

Before the grid runs on all the training athletes, nested cross-validation asks whether that procedure pays. It runs the whole grid inside each of the ten outer folds: the inner folds choose the settings from the outer training part alone, and the outer validation part, which took no part in the choice, scores the result. Its mean is compared with the default's validation F1 on the same folds.

__Step 24:__ Define the grid.

In [ ]:
# PreparedEstimator fits preprocessing inside calibration, when requested.
# CachedSearchCV reuses fits only for identical training rows and labels.
churn_template = CachedSearchCV(
    clone(base_nb),
    {"estimator__var_smoothing": smoothing_values, "estimator__priors": prior_values,
     "calibration": calibration_values},
    scoring="f1", cv=churn_folds, n_jobs=4,
)

__Step 25:__ Estimate the grid with nested cross-validation.

In [ ]:
churn_nested = cross_validate(
    churn_template, X_churn_train, y_churn_train, cv=churn_folds,
    scoring={"f1": "f1", "accuracy": "accuracy"},
    return_train_score=True,
)
churn_nested_rows = []
for metric in ("f1", "accuracy"):
    validation_values = churn_nested[f"test_{metric}"]
    churn_nested_rows.append({
        "metric": metric,
        "outer_train_mean": churn_nested[f"train_{metric}"].mean(),
        "outer_validation_mean": validation_values.mean(),
        "fold_sd": validation_values.std(ddof=1),
    })
print(pd.DataFrame(churn_nested_rows).round(4).to_string(index=False))

**F1 0.8055 over the ten outer folds, with a fold standard deviation of 0.0184.** The fold standard deviation describes variation across these splits. The outer training mean, 0.8104, sits close to the validation mean. The repeated folds share rows, so their scatter is not an independent-sample standard error or proof that overfitting is absent.

__Step 26:__ Set the default's validation F1 on the same folds beside the nested mean, and compare their paired fold scores.

In [ ]:
paired_gain = churn_nested["test_f1"] - nb_cv["test_score"]
print(f"the default on the same folds: F1 {nb_cv['test_score'].mean():.4f}")
print(f"nested minus default: mean {paired_gain.mean():+.4f};"
      f" fold range {paired_gain.min():+.4f} to {paired_gain.max():+.4f};"
      f" grid higher in {(paired_gain > 0).sum()} of {len(paired_gain)} folds")

**The nested mean exceeds the default's 0.7765.** The paired differences show how consistently the grid improves on the same held-out folds. This is a descriptive comparison of overlapping splits, not a significance test.

# <font color='#E8800A'>The grid search</font> <a class="anchor" id="search"></a>
[Back to TOC](#toc)

The grid now runs once on all the training athletes, and its winner is the model this week deploys.

__Step 27:__ Run the grid on all the training athletes.

In [ ]:
churn_search = clone(churn_template).fit(X_churn_train, y_churn_train)

__Step 28:__ List the five best combinations, each beside the test F1 of its refit.

In [ ]:
results = churn_search.cv_results_
ranking = pd.DataFrame({
    "method": [candidate.get("calibration") or "none" for candidate in results["params"]],
    "var_smoothing": [f"{candidate['estimator__var_smoothing']:g}"
                      for candidate in results["params"]],
    "priors": [str(candidate["estimator__priors"])
               for candidate in results["params"]],
    "cv_F1": results["mean_test_score"],
}).nlargest(5, "cv_F1")
ranking["test_F1"] = [
    f1_score(y_churn_test, clone(churn_search.estimator).set_params(
        **clone(results["params"][index], safe=False)).fit(
            X_churn_train, y_churn_train).predict(X_churn_test))
    for index in ranking.index]
print(f"{len(results['params'])} combinations; the five best:")
print(ranking.round(4).to_string())

**The five best of 105 combinations score 0.8075 on the validation folds and 0.8245 on the test rows.** All five use isotonic calibration and `var_smoothing=0.1` and differ only in the prior, so the five priors tie. A prior multiplies every athlete's posterior odds by the same factor, which changes the probability of winning but not the order of the athletes, and isotonic calibration relearns the probability from that order alone. The search keeps the first of the tied rows, `priors=None`.

__Step 29:__ Log the winner's own score: its validation F1 on the same folds.

In [ ]:
# The winner's own score is the best of all of them on the same folds.
winner_churn = cross_validate(
    clone(churn_search.best_estimator_), X_churn_train, y_churn_train,
    cv=churn_folds, scoring="f1", return_train_score=True)
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": "joint search winner",
                   "train": winner_churn["train_score"],
                   "validation": winner_churn["test_score"],
                   "model": churn_search.best_estimator_},
       X_churn_test, y_churn_test, "winner's own score", metric=f1_score, label="F1")

__Step 30:__ Log the winner's nested estimate, beside the same deployed model's test score.

In [ ]:
record(churn_log, {"frame": type(churn_folds).__name__, "rows": len(y_churn_train),
                   "folds": churn_folds.get_n_splits(), "name": "joint search winner",
                   "train": churn_nested["train_f1"],
                   "validation": churn_nested["test_f1"],
                   "model": churn_search.best_estimator_},
       X_churn_test, y_churn_test, "nested estimate", metric=f1_score, label="F1")

**The winner's own validation F1, 0.8075, sits above its nested estimate, 0.8055**, because it was chosen as the best of 105 on the same folds. The board publishes the nested estimate.

# <font color='#E8800A'>The number we stand behind</font> <a class="anchor" id="closing"></a>
[Back to TOC](#toc)

The grid's winner was chosen on the training folds. The closing comparison reports its test score beside the default, and the board publishes that score alongside the nested estimate.

__Step 31:__ Report the winner and the default on the test rows.

In [ ]:
chosen_nb = churn_search.best_estimator_
tuned_f1 = f1_score(y_churn_test, chosen_nb.predict(X_churn_test))
default_nb = PreparedEstimator(FoldPreprocessor(classification_recipe),
                               GaussianNB()).fit(X_churn_train, y_churn_train)
default_f1 = f1_score(y_churn_test, default_nb.predict(X_churn_test))

**On the 800 held-out athletes the winner scores 0.8245 and the default 0.8177**, the test F1 of their rows in the log. On the folds the two are further apart, 0.8055 nested against 0.7765. Neither number is adjusted: the configuration was chosen using validation scores alone. The test scores displayed during the sweeps did not select it.

__Step 32:__ Read three test metrics for the default and the winner: F1, ROC-AUC and balanced accuracy.

In [ ]:
print(f"{'':10s}{'F1':>9s}{'ROC-AUC':>10s}{'balanced acc':>14s}")
for name, model in (("default", default_nb), ("tuned", chosen_nb)):
    predicted = model.predict(X_churn_test)
    print(f"{name:10s}{f1_score(y_churn_test, predicted):9.4f}"
          f"{roc_auc_score(y_churn_test, model.predict_proba(X_churn_test)[:, 1]):10.4f}"
          f"{balanced_accuracy_score(y_churn_test, predicted):14.4f}")

__Step 33:__ Build the course scoreboard with the results earlier weeks saved and the floor.

In [ ]:
# The board: a result an earlier week saved joins it only if it was scored on the
# same data, recipe, rows and folds; the floor labels every athlete with the
# majority outcome.
results_dir = "results"
churn_board = Scoreboard("champions outcome", "F1", context=result_context(
    "../../data/interim/champions.csv", classification_recipe_log,
    y_churn_train.index.tolist(), y_churn_test.index.tolist(),
    outer_folds=outer_folds(churn_folds, X_churn_train, y_churn_train)),
    floor=round(f1_score(y_churn_test, DummyClassifier(strategy="most_frequent").fit(
        X_churn_train, y_churn_train).predict(X_churn_test)), 4))
for week, model, cv, test in results_before(7, "classification", results_dir,
                                             metric=churn_board.metric,
                                             context=churn_board.context):
    churn_board.record(week, model, cv, test)

__Step 34:__ Record this week's result on the board, save it with the fitted winner, and print the board.

In [ ]:
# The default was measured on these same folds, so it is the baseline.
default_cv = smoothing_sweep.set_index("var_smoothing").loc[1e-9, "cv_F1"]
print(f"default over the outer folds: F1 {default_cv:.4f}")

# The row carries two numbers: the nested cross-validated mean first, then the
# winner's held-out score. Its label names the winning configuration.
settings = churn_search.best_params_
smoothing = settings["estimator__var_smoothing"]
priors = settings["estimator__priors"]
chosen_label = (f"Gaussian Naive Bayes (var_smoothing={smoothing:g}, priors={priors},"
                f" calibration={settings.get('calibration') or 'none'})")
churn_board.record(7, chosen_label,
                   round(churn_nested["test_f1"].mean(), 4), round(tuned_f1, 4))
churn_board.save(
    results_dir, 7, "classification", baseline_cv=round(default_cv, 4),
    baseline_test=round(default_f1, 4),
)

# The fitted winner is saved beside its result, so a later week can reload it if
# it ends the course as the best model. Loading needs preprocessing.py on the path.
joblib.dump(chosen_nb, f"{results_dir}/week_07_classification_model.joblib")

print()
print(churn_board.table())

<div class="alert alert-block alert-success">

**Gaussian Naive Bayes joins the board at F1 0.8055 over the ten outer folds and 0.8245 held out.** That is above the floor's 0.7529 and below the logistic regression's 0.8486 and 0.8512. Its label names what the grid chose: `var_smoothing=0.1`, `priors=None` and isotonic calibration. The row and the fitted winner are saved in `results/`. LDA's validation F1 in this week's log, 0.8495, sits beside the board's logistic regression, which fits the same linear form.

</div>

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

1. **Without assumptions the likelihood is a count, and most profiles were never counted:** 431 of the 759 test athletes with no gap have a profile no training athlete shares.
2. **Two of the naive assumptions fail on these athletes:** within a class the numeric columns correlate at up to 0.783, and 18 of the 28 prepared columns take only the values 0 and 1.
3. **Allowing covariance and sharing it across classes with LDA raised validation F1** from 0.7765 to 0.8495.
4. **With ten uneven classes, F1 depends on how the classes are averaged:** micro F1 equals accuracy, 0.596 on the folds, against 0.546 macro and 0.644 weighted.
5. **Calibration changes the probabilities as well as the labels:** isotonic calibration lowered validation confidence from 0.9455 to 0.7409, while F1 rose from 0.7765 to 0.7977.
6. **Nested cross-validation put the grid at 0.8055, above the default's 0.7765 on these folds**, and the grid chose isotonic calibration with `var_smoothing=0.1`.

Exit ticket: a model has a high F1 score. Explain in one sentence why
this does not tell you whether its probabilities are calibrated.

In [ ]:
# Write your answer below. No Python is required.
#


# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

- Hand, D.J. & Yu, K. (2001). Idiot's Bayes: not so stupid after all? *International Statistical Review*, 69(3), 385-398. (Why the assumption fails and the classifier works anyway.)
- Domingos, P. & Pazzani, M. (1997). On the optimality of the simple Bayesian classifier under zero-one loss. *Machine Learning*, 29, 103-130.
- Zhang, H. (2004). The optimality of Naive Bayes. *FLAIRS*.
- Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning*, 2nd ed. Springer, § 4.3 (linear and quadratic discriminant analysis).
- Platt, J.C. (1999). Probabilistic outputs for support vector machines and comparisons to regularized likelihood methods. In A.J. Smola, P. Bartlett, B. Schölkopf & D. Schuurmans (Eds.), *Advances in Large Margin Classifiers*, 61-74. MIT Press. (Sigmoid calibration.)
- Zadrozny, B. & Elkan, C. (2002). Transforming classifier scores into accurate multiclass probability estimates. *Proceedings of the Eighth ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 694-699. (Isotonic calibration.)
- scikit-learn User Guide, [Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html) · [`GaussianNB`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html) · [Linear and Quadratic Discriminant Analysis](https://scikit-learn.org/stable/modules/lda_qda.html) · [`LinearDiscriminantAnalysis`](https://scikit-learn.org/stable/modules/generated/sklearn.discriminant_analysis.LinearDiscriminantAnalysis.html) · [Probability calibration](https://scikit-learn.org/stable/modules/calibration.html) · [`CalibratedClassifierCV`](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibratedClassifierCV.html) · [Multiclass averaging](https://scikit-learn.org/stable/modules/model_evaluation.html#from-binary-to-multiclass-and-multilabel).